In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW ticket_silver_data AS
SELECT
    t.ticket_id,
    COALESCE(t.customer_id, 'N/A') AS customer_id,
    COALESCE(t.customer_name, 'N/A') AS customer_name,
    COALESCE(t.customer_email, 'N/A') AS customer_email,
    COALESCE(t.company_name, 'N/A') AS company_name,
    COALESCE(t.account_type, 'N/A') AS account_type,
    COALESCE(t.customer_segment, 'N/A') AS customer_segment,
    COALESCE(t.industry, 'N/A') AS industry,
    COALESCE(t.billing_contact_email, 'Not Provided') AS billing_contact_email,
    COALESCE(t.account_manager, 'Unassigned') AS account_manager,
    t.monthly_contract_value,
    t.account_created_date,
    COALESCE(t.region, 'N/A') AS region,
    COALESCE(t.subscription_type, 'N/A') AS subscription_type,
    COALESCE(t.customer_tenure_months, 0) AS customer_tenure_months,
    COALESCE(t.service_area, 'N/A') AS service_area,
    COALESCE(t.priority, 'N/A') AS priority,
    COALESCE(t.sla_target_hours, 0) AS sla_target_hours,
    COALESCE(t.status, 'N/A') AS status,
    COALESCE(t.team, 'N/A') AS team,
    COALESCE(t.assigned_to, 'Unassigned') AS assigned_to,
    t.issue_description,
    COALESCE(t.resolution_notes, 'No Notes Provided') AS resolution_notes,
    COALESCE(t.channel, 'N/A') AS channel,
    t.ticket_created_date,
    t.ticket_resolved_date,
    COALESCE(t.first_response_time_hours, 0) AS first_response_time_hours,
    COALESCE(t.resolution_time_hours, 0) AS resolution_time_hours,
    COALESCE(t.escalated, 'N/A') AS escalated,
    COALESCE(t.sla_breached, 'N/A') AS sla_breached,
    COALESCE(t.csat_score, 0) AS csat_score,
    COALESCE(t.issue_complexity_score, 0) AS issue_complexity_score,
    COALESCE(t.previous_tickets, 0) AS previous_tickets,
    COALESCE(t.operating_system, 'N/A') AS operating_system,
    COALESCE(t.browser, 'Unknown') AS browser,
    h.global AS global,
    h.name AS nz_holiday_name,
    c.cleaned_sub_category,
    c.cleaned_category,
    CASE
        WHEN t.ticket_id IS NULL THEN false
        WHEN YEAR(t.ticket_created_date) NOT BETWEEN 1990 AND 2026 THEN false
        WHEN YEAR(t.ticket_resolved_date) NOT BETWEEN 1990 AND 2026 THEN false
        WHEN t.ticket_resolved_date < t.ticket_created_date THEN false
        ELSE true END AS is_valid
FROM techsolve.ticket_bronze.ticket_bronze t
LEFT JOIN techsolve.ticket_bronze.holiday_lookup h
    ON t.ticket_created_date = h.date
LEFT JOIN techsolve.ticket_bronze.category_lookup c
    ON t.category = c.ticket_category
""")

# --- clean silver table ---
spark.sql("""
CREATE OR REPLACE TABLE techsolve.ticket_silver.ticket_silver
USING DELTA AS
SELECT * EXCEPT(is_valid)
FROM ticket_silver_data
WHERE is_valid = true
""")

# --- quarantine table ---
spark.sql("""
CREATE OR REPLACE TABLE techsolve.ticket_silver.ticket_silver_quarantine
USING DELTA AS
SELECT *,
    CASE
        WHEN YEAR(ticket_created_date) NOT BETWEEN 1990 AND 2026 THEN 'Invalid ticket_created_date'
        WHEN YEAR(ticket_resolved_date) NOT BETWEEN 1990 AND 2026 THEN 'Invalid ticket_resolved_date'
        WHEN ticket_resolved_date < ticket_created_date THEN 'Resolved before created'
        ELSE 'Unknown'
    END AS quarantine_reason
FROM ticket_silver_data
WHERE is_valid = false
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]